# BT10 Long Portfolio — Paper Trading Dashboard
**Walk-forward validation · $150,000 · 20 stocks · Inception: Jan 2, 2026**

Run all cells top-to-bottom each trading day. Signal computation is gated to once per day; all other cells are safe to re-run.

In [1]:
# Install dependencies (run once)
%pip install yfinance pandas numpy scipy matplotlib pyarrow

Note: you may need to restart the kernel to use updated packages.


## Imports & Configuration

In [2]:
# %% [markdown]
# # BT10 Long Portfolio — Paper Trading Dashboard
# **Walk-forward validation · $150,000 · 20 stocks · Inception: Jan 2, 2026**
 
import os, json, warnings, smtplib
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from datetime import date, timedelta
from scipy import stats
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from itertools import groupby
warnings.filterwarnings('ignore')
 
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#c9d1d9',
    'text.color':       '#c9d1d9', 'xtick.color':     '#8b949e',
    'ytick.color':      '#8b949e', 'grid.color':      '#21262d',
    'grid.alpha': 0.5,             'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d', 'font.size': 9,
})
C = {'gold':'#d29922','green':'#3fb950','red':'#f85149',
     'blue':'#58a6ff','grey':'#8b949e','purple':'#bc8cff'}
 
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
CFG = dict(
    initial_capital    = 150_000,
    cash_buffer_pct    = 0.005,
    target_holdings    = 20,
    min_pos_pct        = 0.02,
    inception_date     = date(2026, 1, 2),
    benchmark          = 'SPY',
    data_lookback_days = 550,
    sd_thresh          = 2.0,
    lb_1y              = 252,
    lb_3m              = 63,
    cusum_gate         = 2.5,
    cusum_k            = 0.50,
    cusum_h            = 2.0,
    w_rs_sharpe        = 0.20,
    w_slope_1y         = 0.10,
    w_pct_outperf      = 0.20,
    w_pnf              = 0.25,
    w_slope_3m         = 0.15,
    w_z_now            = 0.10,
    slope_cap          = 0.50,
    trail_n            = 0.33,
    trail_min_peak     = 0.05,
    cusum_warn         = 50,
    cusum_alert        = 75,
    trail_warn         = 70,
    trail_alert        = 90,
    spread_large       = 0.0005,
    spread_mid         = 0.0010,
    spread_small       = 0.0015,
    send_email         = False,
    email_to           = 'your@email.com',
    email_from         = 'your@email.com',
    email_password     = '',
    base_dir           = r'C:\Documents\pt_bt10_long_py',
)
CFG['investable']     = CFG['initial_capital'] * (1 - CFG['cash_buffer_pct'])
CFG['target_pos_val'] = CFG['investable'] / CFG['target_holdings']
CFG['f_portfolio']    = os.path.join(CFG['base_dir'], 'pt10_portfolio.json')
CFG['f_prices']       = os.path.join(CFG['base_dir'], 'pt10_prices.parquet')
CFG['f_trades']       = os.path.join(CFG['base_dir'], 'pt10_trades.csv')
os.makedirs(CFG['base_dir'], exist_ok=True)
print(f"Config ready. Dir: {CFG['base_dir']}")

Config ready. Dir: C:\Documents\pt_bt10_long_py


## Universe

In [3]:
UNIVERSE_FILE = os.path.join(CFG['base_dir'], 'universe.json')
if os.path.exists(UNIVERSE_FILE):
    with open(UNIVERSE_FILE) as f:
        uni = json.load(f)
    SP500     = uni['sp500']
    SP400_600 = uni['sp400_600']
    SECTOR    = uni['sector_map']
    print(f"Universe: {len(SP500)} SP500  {len(SP400_600)} SP400/600  "
          f"{len(SECTOR)} sectors")
else:
    raise FileNotFoundError(f"Run setup_universe.py first to create {UNIVERSE_FILE}")
 
ALL_TICKERS = list(dict.fromkeys(SP500 + SP400_600))
 
def get_sector(sym):  return SECTOR.get(sym, 'Unknown')
def get_spread(sym):
    if sym in SP500:     return CFG['spread_large']
    if sym in SP400_600: return CFG['spread_mid']
    return CFG['spread_small']
def exec_price(sym, px_v, side='buy'):
    s = get_spread(sym)
    return px_v*(1+s/2) if side=='buy' else px_v*(1-s/2)

Universe: 503 SP500  1003 SP400/600  1506 sectors


## Price Data

In [4]:
def download_prices(tickers, start, end, cache_file):
    today = date.today()
    if os.path.exists(cache_file):
        cached = pd.read_parquet(cache_file)
        if cached.index[-1].date() >= today - timedelta(days=1):
            print(f"Prices current: {len(cached)} rows  "
                  f"last={cached.index[-1].date()}")
            return cached
    print(f"Downloading {len(tickers)} tickers from {start}...")
    batches = [tickers[i:i+100] for i in range(0, len(tickers), 100)]
    frames  = []
    for i, batch in enumerate(batches):
        if i % 5 == 0: print(f"  Batch {i+1}/{len(batches)}")
        try:
            raw = yf.download(batch, start=start, end=end,
                              auto_adjust=True, progress=False, threads=True)
            frames.append(raw['Close'] if isinstance(
                raw.columns, pd.MultiIndex) else raw)
        except Exception as e:
            print(f"  Batch {i} error: {e}")
    px_out = pd.concat(frames, axis=1)
    px_out = px_out.loc[:, ~px_out.columns.duplicated()]
    px_out.index = pd.to_datetime(px_out.index).normalize()
    px_out = px_out.ffill().dropna(how='all')
    px_out.to_parquet(cache_file)
    print(f"Downloaded: {len(px_out)} rows  last={px_out.index[-1].date()}")
    return px_out
 
px = download_prices(
    tickers=[CFG['benchmark']] + ALL_TICKERS,
    start=date.today() - timedelta(days=CFG['data_lookback_days']),
    end=date.today(), cache_file=CFG['f_prices'])
 
LAST_DATE = px.index[-1].date()
print(f"Last trading day: {LAST_DATE}")

  Batch 1/16


$AL: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$ASGN: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")

2 Failed downloads:
['AL', 'ASGN']: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$IAC: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$CSGS: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$SEE: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$PSTG: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")

4 Failed downloads:
['IAC', 'CSGS', 'SEE',

  Batch 6/16


$MCW: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$BK: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")

2 Failed downloads:
['MCW', 'BK']: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$JHG: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24)
$STEL: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24)

2 Failed downloads:
['JHG', 'STEL']: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24)
$AVNS: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24)
$CPRX: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24)
$HOLX: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$APLS: possibly delisted; n

  Batch 11/16


$GTLS: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24)
$BLD: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24)

2 Failed downloads:
['GTLS', 'BLD']: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24)
$CTRA: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['CTRA']: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$EXPI: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['EXPI']: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$WSR: possibly delisted; no price data found  (1d 2025-02-20 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$CWEN-A: possibly delisted; n

  Batch 16/16
Downloaded: 378 rows  last=2026-08-21
Last trading day: 2026-08-21


## Scoring Functions

In [5]:
def ols_slope(y):
    y = np.asarray(y, dtype=float)
    if len(y) < 2: return 0.0
    return float(np.polyfit(np.arange(len(y), dtype=float), y, 1)[0])
    
 
def ols_project(y):
    y = np.asarray(y, dtype=float); n = len(y)
    if n < 2: return dict(slope=0.0, proj=float(y[-1]), se=1.0, z=0.0)
    t = np.arange(n, dtype=float)
    b, a  = np.polyfit(t, y, 1)
    fitted= a + b*t; proj = a + b*(n-1)
    res   = y - fitted
    se    = float(np.sqrt(np.sum(res**2) / max(n-2, 1)))
    z     = float((y[-1] - proj) / se) if se > 0 else 0.0
    return dict(slope=float(b), proj=float(proj), se=float(se), z=float(z))
 
def safe_prank(arr):
    arr = np.asarray(arr, dtype=float)
    if len(arr) <= 1: return np.full(len(arr), 0.5)
    from scipy.stats import rankdata
    return (rankdata(arr) - 1) / max(len(arr)-1, 1)
 
def compute_metrics(sym, as_of=None):
    if as_of is None: as_of = LAST_DATE
    sub = px.loc[:pd.Timestamp(as_of), :]
    if sym not in sub.columns or CFG['benchmark'] not in sub.columns: return None
    mrg = sub[[sym, CFG['benchmark']]].dropna()
    if len(mrg) < CFG['lb_1y']: return None
    px_s = mrg[sym].values; px_b = mrg[CFG['benchmark']].values
    rs   = px_s / px_b
    n1y  = min(CFG['lb_1y'], len(rs)); n3m = min(CFG['lb_3m'], len(rs))
    rs1y = rs[-n1y:]; rs3m = rs[-n3m:]
    p1y  = ols_project(rs1y)
    if abs(p1y['z']) > CFG['sd_thresh']: return None
    rr   = np.diff(np.log(rs1y))
    rs_sh= rr.mean()*252/(rr.std()*np.sqrt(252)) if rr.std()>0 else 0.0
    dr   = np.diff(np.log(mrg[-n1y:].values), axis=0)
    pct_o= float(np.mean(dr[:,0] > dr[:,1]))
    px1y = px_s[-n1y:]; box = px1y[-1]*0.03
    d    = np.diff(px1y)
    dirs = np.where(d>box,'X',np.where(d<-box,'O','='))
    xc   = sum(1 for k,g in groupby(dirs) if k=='X')
    pnf  = float((px1y[-1]*(1+xc*0.03))/px1y[-1]-1)
    dr3m = np.diff(np.log(mrg[-n3m:].values), axis=0)
    ev   = float((dr3m[:,0]-dr3m[:,1]).std()*np.sqrt(252)) if len(dr3m)>1 else 0.20
    return dict(ticker=sym, last_px=float(px_s[-1]),
                slope_1y=ols_slope(rs1y), slope_3m=ols_slope(rs3m),
                rs_sharpe=float(rs_sh), pct_outperf=pct_o,
                pnf=pnf, sector=get_sector(sym), z_now=float(p1y['z']),
                ols_se=float(p1y['se']), ann_ex_vol=max(ev, 0.05))
 
def score_candidates(metrics_list):
    df = pd.DataFrame([m for m in metrics_list if m is not None])
    if df.empty: return df
    def pr(c): return safe_prank(df[c].values)
    df['slope_1y_pr'] = np.minimum(pr('slope_1y'), CFG['slope_cap'])
    df['composite']   = (
        CFG['w_rs_sharpe']*pr('rs_sharpe') +
        CFG['w_slope_1y']*df['slope_1y_pr'] +
        CFG['w_pct_outperf']*pr('pct_outperf') +
        CFG['w_pnf']*pr('pnf') +
        CFG['w_slope_3m']*pr('slope_3m') +
        CFG['w_z_now']*pr('z_now'))
    return df.sort_values('composite', ascending=False).reset_index(drop=True)
 
def cusum_dn(z_vec):
    S = 0.0
    for z in z_vec:
        if np.isnan(z): continue
        S = min(0.0, S+z+CFG['cusum_k']) if z < -CFG['cusum_gate'] else 0.0
    return S < -CFG['cusum_h'], round(S, 4), round(min(1, abs(S)/CFG['cusum_h'])*100, 1)
 
print("Scoring functions ready.")

Scoring functions ready.


## Portfolio Helpers

In [6]:
def default_position(sym, shares, avg_cost, entry_date, spy_px, ann_ev):
    return dict(
        ticker=sym, sector=get_sector(sym), shares=shares,
        avg_cost=round(avg_cost,4), cost_basis=round(shares*avg_cost,2),
        entry_date=str(entry_date), entry_spy_px=round(spy_px,4),
        current_px=round(avg_cost,4), market_val=round(shares*avg_cost,2),
        
        unrealized=0.0, weight=0.0, days_held=0,
        ann_ex_vol=round(ann_ev,6),
        trail_threshold=round(CFG['trail_n']*ann_ev,6),
        peak_excess=0.0, trail_gap=0.0, trail_pct_to_fire=0.0,
        cusum_z_hist=[], cusum_stat=0.0, cusum_pct=0.0, z_now=0.0)
 
def load_portfolio():
    if not os.path.exists(CFG['f_portfolio']): return None
    with open(CFG['f_portfolio']) as f: port = json.load(f)
    port['inception_date']   = date.fromisoformat(port['inception_date'])
    port['last_signal_date'] = (date.fromisoformat(port['last_signal_date'])
        if port.get('last_signal_date') else None)
    print(f"Portfolio loaded | inception: {port['inception_date']} "
          f"| holdings: {len(port['holdings'])}")
    return port
 
def save_portfolio(port):
    p = port.copy()
    p['inception_date']   = str(p['inception_date'])
    p['last_signal_date'] = str(p['last_signal_date']) \
        if p.get('last_signal_date') else None
    with open(CFG['f_portfolio'], 'w') as f:
        json.dump(p, f, indent=2, default=str)

In [7]:
## Backfill Functions
def backfill_daily_log(port):
    """Populate daily_log for every trading day from inception to today."""
    inc_date = port['inception_date']
    if isinstance(inc_date, str): inc_date = date.fromisoformat(inc_date)
    all_dates = [d.date() for d in px.index if d.date() >= inc_date]
    if not all_dates: return port
    trades    = pd.read_csv(CFG['f_trades'])
    init_buys = trades[trades['action']=='BUY']
    init_sh   = dict(zip(init_buys['ticker'], init_buys['shares'].astype(int)))
    
    init_cash = port['cash']   # use portfolio cash — do not recompute
    print(f"Backfilling {len(all_dates)} days "
          f"({min(all_dates)} → {max(all_dates)})...")
    log_rows = []
    for dt in all_dates:
        dt_ts = pd.Timestamp(dt)
        mv = 0.0
        for sym, shares in init_sh.items():
            if sym not in px.columns: continue
            try:
                v = float(px.loc[dt_ts, sym])
                if not np.isnan(v) and v > 0: mv += shares*v
            except: pass
        spy = None
        try:
            v = float(px.loc[dt_ts, CFG['benchmark']])
            if not np.isnan(v): spy = round(v, 4)
        except: pass
        log_rows.append(dict(date=str(dt),
                             tot_value=round(mv+init_cash,2),
                             cash=round(init_cash,2),
                             spy_px=spy, n_holdings=len(init_sh)))
    sv = [r['spy_px'] for r in log_rows if r['spy_px']]
    if sv:
        print(f"SPY:  {sv[0]:.2f}→{sv[-1]:.2f} = {sv[-1]/sv[0]-1:.1%}  (expect ~11%)")
        print(f"Port: {log_rows[0]['tot_value']:,.0f}→"
              f"{log_rows[-1]['tot_value']:,.0f} "
              f"= {log_rows[-1]['tot_value']/log_rows[0]['tot_value']-1:.1%}")
    port['daily_log'] = log_rows
    return port
 
def backfill_cusum_zscores(port):
    """
    Rebuild cusum_z_hist for each held position for every day since entry.
    Allows CUSUM to fire correctly on first run rather than needing weeks.
    """
    inc_date = port['inception_date']
    if isinstance(inc_date, str): inc_date = date.fromisoformat(inc_date)
    print("Backfilling CUSUM z-score histories...")
    new_sells = []
    for sym, h in port['holdings'].items():
        entry_dt  = date.fromisoformat(h['entry_date'])
        all_dates = [d.date() for d in px.index if d.date() >= entry_dt]
        z_hist    = []
        for dt in all_dates:
            dt_ts = pd.Timestamp(dt)
            sub   = px.loc[:dt_ts, :]
            if sym not in sub.columns: continue
            mrg = sub[[sym, CFG['benchmark']]].dropna()
            if len(mrg) < CFG['lb_1y']: continue
            rs   = (mrg[sym]/mrg[CFG['benchmark']]).values
            n1y  = min(CFG['lb_1y'], len(rs))
            p    = ols_project(rs[-n1y:])
            z_hist.append(p['z'])
        h['cusum_z_hist'] = z_hist
        if len(z_hist) >= 2:
            fired, stat, pct = cusum_dn(z_hist)
            h['cusum_stat'] = stat
            h['cusum_pct']  = pct
            if fired:
                h['exit_type_pending']   = 'CUSUM'
                h['exit_reason_pending'] = f"CUSUM={stat}"
                new_sells.append(sym)
                print(f"  CUSUM SIGNAL: {sym}  stat={stat}")
        print(f"  {sym}: {len(z_hist)} z-obs  cusum={h['cusum_stat']:.3f}")
    if new_sells:
        port['pending_sells'] = list(set(
            port.get('pending_sells', []) + new_sells))
    save_portfolio(port)
    print(f"Done. {len(new_sells)} CUSUM signal(s) queued.")
    return port

## Build Initial Portfolio

In [8]:
def build_initial_portfolio():
    print("Building initial BT10 portfolio...")
    inc    = CFG['inception_date']
    inc_ts = pd.Timestamp(inc)
    metrics = []
    for sym in ALL_TICKERS:
        
        try:
            m = compute_metrics(sym, as_of=inc)
            if m: metrics.append(m)
        except: pass
    ranked = score_candidates(metrics)
    print(f"Candidates: {len(ranked)}")
    selected = ranked.head(CFG['target_holdings'])
    spy_0    = float(px.loc[:inc_ts, CFG['benchmark']].iloc[-1])
    t_val    = CFG['investable'] / CFG['target_holdings']
    min_val  = CFG['initial_capital'] * CFG['min_pos_pct']
    holdings = {}; total_invested = 0.0
    for _, row in selected.iterrows():
        sym    = row['ticker']
        px_0   = float(px.loc[:inc_ts, sym].iloc[-1])
        ep     = exec_price(sym, px_0, 'buy')
        shares = int(max(t_val, min_val) / ep)
        if shares == 0: continue
        holdings[sym] = default_position(sym, shares, ep, inc, spy_0,
                                          row['ann_ex_vol'])
        holdings[sym]['z_now']     = row['z_now']
        holdings[sym]['composite'] = round(row['composite'], 4)
        total_invested += shares * ep
    cash = CFG['initial_capital'] - total_invested
    tot  = sum(h['shares']*h['avg_cost'] for h in holdings.values()) + cash
    for h in holdings.values():
        h['weight'] = round(h['shares']*h['avg_cost']/tot, 4)
    port = dict(
        inception_date=inc, inception_value=CFG['initial_capital'],
        last_signal_date=None, holdings=holdings,
        cash=round(cash,2), total_value=round(tot,2),
        pending_sells=[], pending_buys=[], daily_log=[],
    )
    trades = [dict(date=str(inc), action='BUY', ticker=sym,
                   sector=get_sector(sym), shares=h['shares'],
                   price=h['avg_cost'], value=round(h['shares']*h['avg_cost'],2),
                   exit_type='', reason='Initial BT10 portfolio',
                   portfolio_value=round(tot,2))
              for sym, h in holdings.items()]
    pd.DataFrame(trades).to_csv(CFG['f_trades'], index=False)
    save_portfolio(port)
    print(f"Built: {len(holdings)} holdings | "
          f"invested ${total_invested:,.0f} | cash ${cash:,.0f}")
    port = backfill_daily_log(port)
    save_portfolio(port)
    return port


## Load or Build Portfolio

In [9]:
port = load_portfolio()
if port is None:
    port = build_initial_portfolio()
 
# Backfill CUSUM histories if empty (first run after build)

_needs_cusum = any(
    len(h.get('cusum_z_hist', [])) == 0
    for h in port['holdings'].values())
if _needs_cusum:
    print("CUSUM histories empty — backfilling...")
    port = backfill_cusum_zscores(port)

Portfolio loaded | inception: 2026-01-02 | holdings: 20


In [ ]:
# ── Fetch split history ONCE before the replay loop ──────────────────────────
# Yahoo returns splits as a Series indexed by ex-date, value = split factor
# (e.g. 2.0 for a 2-for-1 split, 0.5 for a 1-for-2 reverse split).
# We build a dict: {ticker: Series of splits} for every ticker we might hold.
print("Pre-fetching split history for all tickers...")
split_history = {}
for sym in ALL_TICKERS + [CFG['benchmark']]:
    try:
        s = yf.Ticker(sym).splits
        if s is not None and len(s) > 0:
            s.index = s.index.tz_localize(None) if s.index.tz else s.index
            split_history[sym] = s
    except Exception:
        pass
print(f"  Split history loaded for "
      f"{sum(1 for v in split_history.values() if len(v)>0)} tickers.")


def apply_splits_for_day(sim_long, sim_short, dt_ts):
    """
    Called once at the top of each replay day. For every held position,
    checks whether a split ex-date falls on exactly this day and adjusts
    shares + per-share cost in place if so.

    Why ex-date only, not a range:
    The day loop already processes days in order; we only need to check
    if TODAY is a split date, because every prior day was already checked
    on its own pass. This is O(1) per call rather than O(range).

    Adjustment logic:
      shares_new   = round(shares_old * factor)
      avg_cost_new = avg_cost_old / factor        (total cost basis unchanged)

    Handles reverse splits (factor < 1.0) identically.
    """
    adjustments = []

    for sym, h in sim_long.items():
        if sym not in split_history:
            continue
        splits_on_day = split_history[sym].get(dt_ts, None)
        if splits_on_day is None or abs(float(splits_on_day) - 1.0) < 1e-6:
            continue
        factor = float(splits_on_day)

        old_shares = h['shares']
        old_cost   = h['avg_cost']
        h['shares']   = int(round(old_shares * factor))
        h['avg_cost']  = round(old_cost / factor, 6)

        # Trail stop watermarks are price-based -- must also adjust
        if 'peak_price' in h:
            h['peak_price'] = round(h['peak_price'] / factor, 6)
        if 'peak_excess' in h:
            # peak_excess = cumulative excess return over SPY, not a price,
            # so it does NOT need adjusting -- it's a dimensionless return
            pass
        # entry_spy_px is SPY's price at entry -- not affected by stock split
        # avg_cost used in excess-return calculation: stk_ret = px_now/avg_cost - 1
        # avg_cost_new = avg_cost_old / factor is correct since px_now is
        # already split-adjusted by Yahoo, so the return is preserved

        adjustments.append(
            f"  [SPLIT] LONG {sym}: {factor:.4f}x  "
            f"shares {old_shares}→{h['shares']}  "
            f"avg_cost ${old_cost:.4f}→${h['avg_cost']:.4f}"
        )

    for sym, h in sim_short.items():
        if sym not in split_history:
            continue
        splits_on_day = split_history[sym].get(dt_ts, None)
        if splits_on_day is None or abs(float(splits_on_day) - 1.0) < 1e-6:
            continue
        factor = float(splits_on_day)

        old_shares = h['shares']
        old_sp     = h['short_price']
        h['shares']      = int(round(old_shares * factor))
        h['short_price'] = round(old_sp / factor, 6)
        # short P&L = shares * (short_price - px_now)
        # after split: shares*factor * (short_price/factor - px_now/factor)
        #            = shares * (short_price - px_now)  -- P&L unchanged ✓

        adjustments.append(
            f"  [SPLIT] SHORT {sym}: {factor:.4f}x  "
            f"shares {old_shares}→{h['shares']}  "
            f"short_price ${old_sp:.4f}→${h['short_price']:.4f}"
        )

    for msg in adjustments:
        print(msg)

Pre-fetching split history for all tickers...


$AL: possibly delisted; no price data found  (1d 1927-09-18 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$ASGN: possibly delisted; no price data found  (1d 1927-09-18 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$CSGS: possibly delisted; no price data found  (1d 1927-09-18 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$IAC: possibly delisted; no price data found  (1d 1927-09-18 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$PSTG: possibly delisted; no price data found  (1d 1927-09-18 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$SEE: possibly delisted; no price data found  (1d 1927-09-18 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$AMWD: possibly delisted; no price data found  (1d 1927-09-18 -> 2026-08-24) (Yahoo error = "No data found, symbol may be delisted")
$GDEN: possibly delisted; no price data found  (1d 1927-09-18 -> 2026-08-

In [ ]:
def backfill_signals(port, force=False):
    """
    Replay every trading day from inception to yesterday, evaluating trail
    and CUSUM exit signals, executing sells, buying replacements, and
    writing a correct daily_log.

    Runs once — gated on port['backfill_complete']. Pass force=True to re-run.
    """
    if port.get('backfill_complete') and not force:
        
        print("Backfill already complete. Pass force=True to re-run.")
        return port

    inc_date = port['inception_date']
    if isinstance(inc_date, str):
        inc_date = date.fromisoformat(inc_date)

    yesterday   = LAST_DATE - timedelta(days=1)
    replay_days = [d.date() for d in px.index
                   if inc_date <= d.date() <= yesterday]

    if not replay_days:
        print("No days to replay.")
        return port

    print(f"Replaying {len(replay_days)} trading days "
          f"({replay_days[0]} → {replay_days[-1]})...")

    # ── Rebuild port state at inception from the trade log ────────────────────
    trades_df  = pd.read_csv(CFG['f_trades'])
    init_buys  = trades_df[trades_df['action'] == 'BUY'].copy()
    inc_buys   = init_buys[
        init_buys['date'] == str(inc_date)
    ]  # only Jan 2 buys

    inc_ts = pd.Timestamp(inc_date)
    spy_0  = float(px.loc[:inc_ts, CFG['benchmark']].iloc[-1])

    # Reconstruct holdings at inception
    sim_holdings = {}
    for _, row in inc_buys.iterrows():
        sym    = row['ticker']
        shares = int(row['shares'])
        cost   = float(row['price'])
        m0     = compute_metrics(sym, as_of=inc_date)
        ev     = m0['ann_ex_vol'] if m0 else 0.20
        h      = default_position(sym, shares, cost, inc_date, spy_0, ev)
        sim_holdings[sym] = h

    total_invested = sum(h['shares'] * h['avg_cost']
                         for h in sim_holdings.values())
    sim_cash       = CFG['initial_capital'] - total_invested
    sim_log        = []
    sim_trades     = []   # additional trades beyond the initial buys

    # ── Day-by-day replay ─────────────────────────────────────────────────────
    for dt in replay_days:
        dt_ts   = pd.Timestamp(dt)

        # ── Split adjustment (must be first, before any price comparisons) ────
        apply_splits_for_day(sim_long, dt_ts)
        
        spy_now = None
        try:
            v = float(px.loc[dt_ts, CFG['benchmark']])
            if not np.isnan(v): spy_now = v
        except:
            pass

        # ── Trail + CUSUM signal check ────────────────────────────────────────
        new_sells = []
        for sym, h in sim_holdings.items():
            if sym not in px.columns:
                continue
            try:
                px_now = float(px.loc[dt_ts, sym])
            except:
                continue
            if np.isnan(px_now) or px_now <= 0:
                continue

            exit_f = False; exit_t = ''; exit_r = ''

            # Trail
            if spy_now and h['entry_spy_px'] > 0:
                stk_ret = px_now / h['avg_cost'] - 1
                spy_ret = spy_now / h['entry_spy_px'] - 1
                ex_now  = stk_ret - spy_ret
                h['peak_excess'] = max(h['peak_excess'], ex_now)
                gap = h['peak_excess'] - ex_now
                thr = h['trail_threshold']
                h['trail_gap'] = round(gap, 6)
                h['trail_pct_to_fire'] = (
                    round(min(gap / thr, 1.0) * 100, 1)
                    if (thr > 0 and h['peak_excess'] > CFG['trail_min_peak'])
                    else 0.0)
                if h['peak_excess'] > CFG['trail_min_peak'] and gap > thr:
                    exit_f = True; exit_t = 'TRAIL'
                    exit_r = (f"TRAIL: peak={h['peak_excess']:.1%} "
                              f"gap={gap:.1%} thresh={thr:.1%}")

            # CUSUM
            if not exit_f:
                m = compute_metrics(sym, as_of=dt)
                if m:
                    h['z_now'] = round(m['z_now'], 4)
                    h['cusum_z_hist'].append(m['z_now'])
                if len(h['cusum_z_hist']) >= 2:
                    fired, stat, pct = cusum_dn(h['cusum_z_hist'])
                    h['cusum_stat'] = stat; h['cusum_pct'] = pct
                    if fired:
                        exit_f = True; exit_t = 'CUSUM'
                        exit_r = f"CUSUM={stat} < -{CFG['cusum_h']}"

            if exit_f:
                h['exit_type_pending']   = exit_t
                h['exit_reason_pending'] = exit_r
                new_sells.append(sym)

        # ── Execute sells ─────────────────────────────────────────────────────
        sold_today = set()
        for sym in new_sells:
            if sym not in sim_holdings:
                continue
            h = sim_holdings[sym]
            try:
                px_now = float(px.loc[dt_ts, sym])
            except:
                continue
            ep         = exec_price(sym, px_now, 'sell')
            sim_cash  += h['shares'] * ep
            sim_trades.append(dict(
                date=str(dt), action='SELL', ticker=sym,
                sector=get_sector(sym), shares=h['shares'],
                price=round(ep, 4), value=round(h['shares'] * ep, 2),
                exit_type=h.get('exit_type_pending', ''),
                reason=h.get('exit_reason_pending', ''),
                portfolio_value=0.0))   # filled below after mtm
            del sim_holdings[sym]
            sold_today.add(sym)
            print(f"  {dt} SELL [{h.get('exit_type_pending','')}]: {sym}")

        # ── Queue and execute replacements ────────────────────────────────────
        if sold_today:
            held     = set(sim_holdings.keys())
            excluded = held | sold_today
            cand_m   = [compute_metrics(s, as_of=dt)
                        for s in ALL_TICKERS if s not in excluded]
            cand_sc  = score_candidates([m for m in cand_m if m])
            n_need   = CFG['target_holdings'] - len(sim_holdings)
            buys     = (cand_sc.head(n_need)['ticker'].tolist()
                        if n_need > 0 and not cand_sc.empty else [])

            for sym in buys:
                m = compute_metrics(sym, as_of=dt)
                if not m:
                    continue
                try:
                    px_now = float(px.loc[dt_ts, sym])
                except:
                    continue
                if np.isnan(px_now) or px_now <= 0:
                    continue
                ep      = exec_price(sym, px_now, 'buy')
                cv      = (sum(h['shares'] * px_now
                               for h in sim_holdings.values()
                               if sym not in px.columns) + sim_cash)
                # simple sizing: equal-weight on current portfolio value
                cv_est  = (sum(
                               float(px.loc[dt_ts, s])
                               * sim_holdings[s]['shares']
                               for s in sim_holdings
                               if s in px.columns) + sim_cash)
                t_val   = cv_est * (1 - CFG['cash_buffer_pct']) / CFG['target_holdings']
                min_val = cv_est * CFG['min_pos_pct']
                if sim_cash < min_val:
                    continue
                shares = int(min(max(t_val, min_val),
                                 sim_cash * 0.95) / ep)
                if shares == 0:
                    continue
                cost       = shares * ep
                sim_cash  -= cost
                spy_entry  = spy_now if spy_now else 0.0
                sim_holdings[sym] = default_position(
                    sym, shares, ep, dt, spy_entry, m['ann_ex_vol'])
                sim_holdings[sym]['z_now'] = m['z_now']
                sim_trades.append(dict(
                    date=str(dt), action='BUY', ticker=sym,
                    sector=get_sector(sym), shares=shares,
                    price=round(ep, 4), value=round(cost, 2),
                    exit_type='', reason='Replacement buy',
                    portfolio_value=0.0))
                print(f"  {dt} BUY (replacement): {sym}")

        # ── Daily log entry ───────────────────────────────────────────────────
        mv = 0.0
        for sym, h in sim_holdings.items():
            if sym not in px.columns:
                continue
            try:
                v = float(px.loc[dt_ts, sym])
                if not np.isnan(v) and v > 0:
                    mv += h['shares'] * v
            except:
                pass
        tot_val = round(mv + sim_cash, 2)

        # back-fill portfolio_value in today's trades
        for t in sim_trades:
            if t['portfolio_value'] == 0.0 and t['date'] == str(dt):
                t['portfolio_value'] = tot_val

        sim_log.append(dict(
            date=str(dt),
            tot_value=tot_val,
            cash=round(sim_cash, 2),
            spy_px=round(spy_now, 4) if spy_now else None,
            n_holdings=len(sim_holdings)))

    # ── Commit results back to port ───────────────────────────────────────────
    port['holdings']         = sim_holdings
    port['cash']             = round(sim_cash, 2)
    port['daily_log']        = sim_log
    port['last_signal_date'] = str(replay_days[-1])
    port['backfill_complete']= True

    # Rewrite trade log: keep original init buys, append replay trades
    init_rows = trades_df[trades_df['date'] == str(inc_date)]
    all_trades = pd.concat(
        [init_rows, pd.DataFrame(sim_trades)], ignore_index=True)
    all_trades.to_csv(CFG['f_trades'], index=False)

    save_portfolio(port)

    n_sells = sum(1 for t in sim_trades if t['action'] == 'SELL')
    n_buys  = sum(1 for t in sim_trades if t['action'] == 'BUY')
    print(f"\nBackfill complete: {len(replay_days)} days replayed | "
          f"{n_sells} sell(s) | {n_buys} buy(s) | "
          f"{len(sim_holdings)} holdings now | "
          f"cash ${sim_cash:,.0f}")
    return port


# ── Run once (remove backfill_complete key from portfolio JSON to re-run) ─────
if not port.get('backfill_complete'):
    print("Backfill needed — replaying inception → yesterday...")
    port = backfill_signals(port)
else:
    print(f"Backfill complete. Holdings: {len(port['holdings'])}  "
          f"Last signal: {port.get('last_signal_date')}")

## Mark-to-Market
*Always runs — updates current prices and unrealized P&L.*

In [ ]:
def mtm(port):
    last_ts  = pd.Timestamp(LAST_DATE)
    spy_now  = float(px.loc[last_ts, CFG['benchmark']]) \
        if last_ts in px.index else None
    total_mv = 0.0
    
    for sym, h in port['holdings'].items():
        if sym not in px.columns: continue
        try: px_now = float(px.loc[last_ts, sym])
        except: continue
        if np.isnan(px_now) or px_now <= 0: continue
        h['current_px'] = round(px_now, 4)
        h['market_val'] = round(h['shares']*px_now, 2)
        h['unrealized'] = round(h['shares']*(px_now-h['avg_cost']), 2)
        h['days_held']  = (LAST_DATE-date.fromisoformat(h['entry_date'])).days
        total_mv += h['market_val']
        if spy_now and h['entry_spy_px'] > 0:
            stk_ret = px_now/h['avg_cost']-1
            spy_ret = spy_now/h['entry_spy_px']-1
            ex_now  = stk_ret - spy_ret
            h['peak_excess'] = max(h['peak_excess'], ex_now)
            gap = h['peak_excess'] - ex_now
            thr = h['trail_threshold']
            h['trail_gap'] = round(gap, 6)
            # FIX: only show trail% when stop is armed (peak > min_peak)
            h['trail_pct_to_fire'] = round(
                min(gap/thr,1.0)*100, 1) \
                if (thr>0 and h['peak_excess']>CFG['trail_min_peak']) \
                else 0.0
    port['total_value'] = round(total_mv + port['cash'], 2)
    for h in port['holdings'].values():
        h['weight'] = round(h['market_val']/port['total_value'], 4) \
            if port['total_value'] > 0 else 0
    return port
 
port = mtm(port)
tot_ret = (port['total_value']-port['inception_value'])/port['inception_value']
print(f"MTM done | value: ${port['total_value']:,.0f} | return: {tot_ret:.1%}")

## Refresh Display Metrics
*Always runs — updates CUSUM% and Trail% for the holdings table without triggering trades.*

In [ ]:
# %% ── Refresh display metrics (always runs) ───────────────────────────────────
# Updates cusum_pct and trail_pct_to_fire for the holdings table
# without triggering any trades — safe to run every session
last_ts = pd.Timestamp(LAST_DATE)
spy_now = float(px.loc[last_ts, CFG['benchmark']]) \
          if last_ts in px.index else None

for sym, h in port['holdings'].items():
    # Refresh CUSUM display
    zh = [float(z) for z in h.get('cusum_z_hist', [])]
    if len(zh) >= 2:
        _, stat, pct = cusum_dn(zh)
        
        h['cusum_stat'] = stat
        h['cusum_pct']  = pct

    # Refresh trail display
    px_now = h.get('current_px', 0)
    if spy_now and px_now and h['entry_spy_px'] > 0:
        stk_ret = px_now / h['avg_cost'] - 1
        spy_ret = spy_now / h['entry_spy_px'] - 1
        ex_now  = stk_ret - spy_ret
        h['peak_excess'] = max(h['peak_excess'], ex_now)
        gap = h['peak_excess'] - ex_now
        thr = h['trail_threshold']
        h['trail_gap'] = round(gap, 6)
        h['trail_pct_to_fire'] = round(
            min(gap/thr, 1.0)*100, 1) \
            if (thr > 0 and h['peak_excess'] > CFG['trail_min_peak']) \
            else 0.0

save_portfolio(port)
print("Display metrics refreshed.")

## Correct Peak Excess from Full Price History
*Always runs — scans full price history since entry to find true peak excess return.*

In [ ]:
# Fix peak_excess by scanning full history since entry
last_ts = pd.Timestamp(LAST_DATE)
spy_now = float(px.loc[last_ts, CFG['benchmark']])

print("Scanning historical peak excess for all holdings...\n")

for sym, h in port['holdings'].items():
    entry_dt = date.fromisoformat(h['entry_date'])
    if sym not in px.columns: continue
    
    ep  = h['avg_cost']
    esp = h['entry_spy_px']
    if ep <= 0 or esp <= 0: continue

    trade_dates = [d for d in px.index if d.date() >= entry_dt]
    peak = 0.0
    for dt in trade_dates:
        try:
            pxd  = float(px.loc[dt, sym])
            spyd = float(px.loc[dt, CFG['benchmark']])
            if np.isnan(pxd) or np.isnan(spyd) or pxd <= 0: continue
            ex   = (pxd/ep - 1) - (spyd/esp - 1)
            peak = max(peak, ex)
        except: continue

    px_now  = h['current_px']
    ex_now  = (px_now/ep - 1) - (spy_now/esp - 1)
    gap     = peak - ex_now
    thr     = h['trail_threshold']

    h['peak_excess']      = round(peak, 6)
    h['trail_gap']        = round(gap, 6)
    h['trail_pct_to_fire']= round(
        min(gap/thr, 1.0)*100, 1) \
        if (thr > 0 and peak > CFG['trail_min_peak']) else 0.0

    print(f"  {sym:6}: peak={peak:.1%}  ex_now={ex_now:.1%}  "
          f"gap={gap:.1%}  thr={thr:.1%}  trail={h['trail_pct_to_fire']:.1f}%")

save_portfolio(port)
print("\nPeak excess corrected. Re-run execute_pending() to process any fires.")

## Queue Trail Fires After Peak Correction
*Runs after peak correction — queues any trail sells that fire after the corrected peak.*

In [ ]:
# Immediately check for trail fires on corrected peak excess values
last_ts = pd.Timestamp(LAST_DATE)
spy_now = float(px.loc[last_ts, CFG['benchmark']])
new_sells = []

for sym, h in port['holdings'].items():
    px_now = h['current_px']
    if not px_now or px_now <= 0: continue
    if spy_now and h['entry_spy_px'] > 0:
        gap = h['peak_excess'] - ((px_now/h['avg_cost']-1) -
                                  
                                   (spy_now/h['entry_spy_px']-1))
        thr = h['trail_threshold']
        if h['peak_excess'] > CFG['trail_min_peak'] and gap > thr:
            h['exit_type_pending']   = 'TRAIL'
            h['exit_reason_pending'] = (
                f"TRAIL: peak={h['peak_excess']:.1%} "
                f"gap={gap:.1%} thresh={thr:.1%}")
            new_sells.append(sym)
            print(f"  TRAIL FIRE: {sym}  peak={h['peak_excess']:.1%}  "
                  f"gap={gap:.1%}  thr={thr:.1%}")

port['pending_sells'] = new_sells
# Reset signals_done so run_signals() fires today to get CUSUM too
port['last_signal_date'] = None
save_portfolio(port)
print(f"\n{len(new_sells)} trail sell(s) queued. "
      f"Re-run all cells from MTM downward.")

## Execute Pending Orders
*Always runs — executes any queued sells and buys.*

In [ ]:
def execute_pending(port):
    last_ts    = pd.Timestamp(LAST_DATE)
    spy_now    = float(px.loc[last_ts, CFG['benchmark']]) \
                 if last_ts in px.index else None
    trade_rows = []
    sold_today = set()

    # ── Sells ──────────────────────────────────────────────────────────────────
    for sym in list(port['pending_sells']):
        if sym not in port['holdings']: continue
        h = port['holdings'][sym]
        
        try: px_now = float(px.loc[last_ts, sym])
        except: continue
        ep  = exec_price(sym, px_now, 'sell')
        pnl = h['shares'] * (ep - h['avg_cost'])
        port['cash'] += h['shares'] * ep
        trade_rows.append(dict(
            date=str(LAST_DATE), action='SELL', ticker=sym,
            sector=get_sector(sym), shares=h['shares'],
            price=round(ep,4), value=round(h['shares']*ep,2),
            exit_type=h.get('exit_type_pending',''),
            reason=h.get('exit_reason_pending',''),
            portfolio_value=round(port['total_value'],2)))
        del port['holdings'][sym]
        sold_today.add(sym)
        print(f"  EXECUTED SELL: {sym}  P&L: ${pnl:,.0f}  "
              f"[{h.get('exit_type_pending','')}]")
    port['pending_sells'] = []

    # ── Queue replacements excluding just-sold tickers ─────────────────────────
    if sold_today:
        held     = set(port['holdings'].keys())
        excluded = held | sold_today
        cand_m   = [compute_metrics(s) for s in ALL_TICKERS
                    if s not in excluded]
        cand_sc  = score_candidates([m for m in cand_m if m])
        n_need   = CFG['target_holdings'] - len(port['holdings'])
        if n_need > 0 and not cand_sc.empty:
            port['pending_buys'] = cand_sc.head(n_need)['ticker'].tolist()
            print(f"  Queued {n_need} replacement(s): {port['pending_buys']}")

    # ── Buys ───────────────────────────────────────────────────────────────────
    for sym in list(port['pending_buys']):
        if sym in port['holdings']: continue
        m = compute_metrics(sym)
        if not m: continue
        try: px_now = float(px.loc[last_ts, sym])
        except: continue
        if not px_now or px_now <= 0: continue
        ep     = exec_price(sym, px_now, 'buy')
        cv     = sum(h['market_val'] for h in port['holdings'].values()) \
                 + port['cash']
        t_val  = cv * (1 - CFG['cash_buffer_pct']) / CFG['target_holdings']
        min_val= cv * CFG['min_pos_pct']
        if port['cash'] < min_val: continue
        shares = int(min(max(t_val, min_val), port['cash'] * 0.95) / ep)
        if shares == 0: continue
        cost   = shares * ep
        port['cash'] -= cost
        sp0    = float(spy_now) if spy_now else 0.0
        port['holdings'][sym] = default_position(
            sym, shares, ep, LAST_DATE, sp0, m['ann_ex_vol'])
        port['holdings'][sym]['z_now'] = m['z_now']
        trade_rows.append(dict(
            date=str(LAST_DATE), action='BUY', ticker=sym,
            sector=get_sector(sym), shares=shares,
            price=round(ep,4), value=round(cost,2),
            exit_type='', reason='Replacement buy',
            portfolio_value=round(port['total_value'],2)))
        print(f"  EXECUTED BUY: {sym}  {shares} @ ${ep:.2f}")
    port['pending_buys'] = []

    # ── Save ───────────────────────────────────────────────────────────────────
    if trade_rows:
        nr = pd.DataFrame(trade_rows)
        if os.path.exists(CFG['f_trades']):
            nr = pd.concat([pd.read_csv(CFG['f_trades']), nr],
                           ignore_index=True)
        nr.to_csv(CFG['f_trades'], index=False)
        port = mtm(port)
        save_portfolio(port)
        print(f"  {len(trade_rows)} order(s) executed and saved.")
    return port

port = execute_pending(port)

## Signal Computation
*Runs once per trading day — checks exit signals for all holdings and scores replacements.*

In [ ]:
signals_done = (port.get('last_signal_date') is not None and
    (port['last_signal_date'] == LAST_DATE
     if isinstance(port['last_signal_date'], date)
     else date.fromisoformat(str(port['last_signal_date'])) == LAST_DATE))


if signals_done:
    print(f"Signals already computed for {LAST_DATE}")
    held   = set(port['holdings'].keys())
    cand_m = [compute_metrics(s) for s in ALL_TICKERS if s not in held]
    cand_sc= score_candidates([m for m in cand_m if m])
else:
    print(f"Computing signals for {LAST_DATE}...")
    last_ts = pd.Timestamp(LAST_DATE)
    spy_now = float(px.loc[last_ts, CFG['benchmark']]) \
        if last_ts in px.index else None
    new_sells = []
 
    for sym, h in port['holdings'].items():
        exit_f = False; exit_t = ''; exit_r = ''
        px_now = h['current_px']
 
        # Trailing stop
        if spy_now and h['entry_spy_px'] > 0:
            stk_ret = px_now/h['avg_cost']-1
            spy_ret = spy_now/h['entry_spy_px']-1
            ex_now  = stk_ret-spy_ret
            h['peak_excess'] = max(h['peak_excess'], ex_now)
            gap = h['peak_excess']-ex_now; thr = h['trail_threshold']
            h['trail_gap'] = round(gap, 6)
            h['trail_pct_to_fire'] = round(
                min(gap/thr,1.0)*100, 1) \
                if (thr>0 and h['peak_excess']>CFG['trail_min_peak']) \
                else 0.0
            if h['peak_excess']>CFG['trail_min_peak'] and gap>thr:
                exit_f=True; exit_t='TRAIL'
                exit_r=(f"TRAIL: peak={h['peak_excess']:.1%} "
                        f"gap={gap:.1%} thresh={thr:.1%}")
 
        # CUSUM
        if not exit_f:
            m = compute_metrics(sym)
            if m:
                h['z_now'] = round(m['z_now'], 4)
                h['cusum_z_hist'].append(m['z_now'])
            if len(h['cusum_z_hist']) >= 2:
                fired, stat, pct = cusum_dn(h['cusum_z_hist'])
                h['cusum_stat'] = stat; h['cusum_pct'] = pct
                if fired:
                    exit_f=True; exit_t='CUSUM'
                    exit_r=f"CUSUM={stat} < -{CFG['cusum_h']}"
 
        if exit_f:
            h['exit_type_pending']   = exit_t
            h['exit_reason_pending'] = exit_r
            new_sells.append(sym)
            print(f"  EXIT SIGNAL [{exit_t}]: {sym}")
 
    port['pending_sells'] = new_sells
 
    # Score replacements — exclude current holdings AND just-sold tickers
    held     = set(port['holdings'].keys())
    just_sold= set(new_sells)   # do not immediately re-buy what we just sold
    excluded = held | just_sold
    cand_m   = [compute_metrics(s) for s in ALL_TICKERS if s not in excluded]
    cand_sc  = score_candidates([m for m in cand_m if m])
    n_need   = CFG['target_holdings'] - len(port['holdings']) + len(new_sells)
    if n_need > 0 and not cand_sc.empty:
        port['pending_buys'] = cand_sc.head(n_need)['ticker'].tolist()
 
    # Daily log
    spy_today = float(px.loc[last_ts, CFG['benchmark']]) \
        if last_ts in px.index else None
    if str(LAST_DATE) not in [r['date'] for r in port['daily_log']]:
        port['daily_log'].append(dict(
            date=str(LAST_DATE),
            tot_value=round(port['total_value'],2),
            cash=round(port['cash'],2),
            spy_px=round(spy_today,4) if spy_today else None,
            n_holdings=len(port['holdings'])))
 
    port['last_signal_date'] = LAST_DATE
    save_portfolio(port)
    print(f"Signals done | {len(new_sells)} sell(s) | "
          f"{len(port['pending_buys'])} buy slot(s)")
    # Execute any signals that just fired immediately
    port = execute_pending(port)


## Performance Series

In [ ]:
dl = (pd.DataFrame(port['daily_log'])
      .assign(date=lambda d: pd.to_datetime(d['date']),
              tot_value=lambda d: d['tot_value'].astype(float),
              spy_px   =lambda d: d['spy_px'].astype(float))
    
      .sort_values('date').reset_index(drop=True))
dl['ret_port'] = dl['tot_value'].pct_change().fillna(0)
dl['ret_spy']  = dl['spy_px'].pct_change().fillna(0)
print(f"Performance: {len(dl)} rows  "
      f"({dl['date'].iloc[0].date()} → {dl['date'].iloc[-1].date()})")
 
if not signals_done:
    held   = set(port['holdings'].keys())
    cand_m = [compute_metrics(s) for s in ALL_TICKERS if s not in held]
    cand_sc= score_candidates([m for m in cand_m if m])

## Summary

In [ ]:
def period_stats(df_sub, base_value=None):
    r = df_sub['ret_port'].values[1:]
    
    s = df_sub['ret_spy'].values[1:]
    if len(r) == 0: return None, None, None
    tot  = float(np.prod(1+r)-1)
    tots = float(np.prod(1+s)-1)
    # Override total return to use capital as base (for since-inception periods)
    if base_value is not None and len(df_sub) > 0:
        last_val = float(df_sub['tot_value'].iloc[-1])
        tot  = (last_val - base_value) / base_value
        tots = float(dl['spy_px'].iloc[-1]/dl['spy_px'].iloc[0]-1)
    n   = len(r)
    ann = float((1+tot)**(252/n)-1) if n >= 2 else tot
    vol = float(r.std()*np.sqrt(252))  if n >= 2 else None
    sh  = ann/vol if (vol and vol > 0) else None
    return tot, tots, sh
 
tot_ret  = (port['total_value']-port['inception_value'])/port['inception_value']
spy_inc  = dl['spy_px'].iloc[-1]/dl['spy_px'].iloc[0]-1
n_sell   = len(port['pending_sells'])
n_buy    = len(port['pending_buys'])
 
print("=" * 60)
print(f"  BT10 LONG PORTFOLIO  |  {LAST_DATE}")
print("=" * 60)
print(f"  Portfolio Value:  ${port['total_value']:>12,.0f}")
print(f"  Total Return:     {tot_ret:>12.1%}")
print(f"  vs SPY:           {tot_ret-spy_inc:>12.1%}  (SPY: {spy_inc:.1%})")
print(f"  Cash:             ${port['cash']:>12,.0f}")
print(f"  Holdings:         {len(port['holdings'])} / {CFG['target_holdings']}")
print(f"  Inception:        {port['inception_date']}")
print("-" * 60)
 
today_m = str(LAST_DATE)[:7]; today_y = str(LAST_DATE)[:4]
periods = {
    'Today':           dl[dl['date'].dt.date == LAST_DATE],
    'Month to Date':   dl[dl['date'].dt.strftime('%Y-%m') == today_m],
    'Year to Date':    dl[dl['date'].dt.strftime('%Y') == today_y],
    'Since Inception': dl,
}
print(f"\n{'Period':<20} {'Return':>8} {'vs SPY':>8} {'Sharpe':>8}")
print("-" * 46)
for label, sub in periods.items():
    # Use capital as base for periods that cover inception year
    use_base = label in ('Since Inception', 'Year to Date')
    base = port['inception_value'] if use_base else None
    tot, tots, sh = period_stats(sub, base_value=base)
    if tot is None:
        print(f"{label:<20} {'—':>8} {'—':>8} {'—':>8}")
    else:
        sh_s = f"{sh:.3f}" if sh else "—"
        print(f"{label:<20} {tot:>8.1%} {tot-tots:>8.1%} {sh_s:>8}")
 
if n_sell: print(f"\n⚠ SELL TODAY:  {', '.join(port['pending_sells'])}")
if n_buy:  print(f"→ BUY TODAY:   {', '.join(port['pending_buys'])}")

## Holdings Table

In [ ]:
h_rows = []
for sym, h in port['holdings'].items():
    pnl_pct = h['unrealized']/h['cost_basis'] if h['cost_basis'] else 0
    max_sig  = max(h['cusum_pct'], h['trail_pct_to_fire'])
    h_rows.append({'Ticker':sym,'Sector':h['sector'],'Shares':h['shares'],
        'AvgCost':h['avg_cost'],'Current':h['current_px'],
        'P&L%':f"{pnl_pct:.1%}",'Wt':f"{h['weight']:.1%}",
        'Z':round(h['z_now'],2),'CUSUM%':f"{h['cusum_pct']:.0f}%",
        'Trail%':f"{h['trail_pct_to_fire']:.0f}%",
        'PeakEx':f"{h['peak_excess']:.1%}",'Days':h['days_held'],
        '_sig':max_sig})
h_df = (pd.DataFrame(h_rows).sort_values('_sig',ascending=False)
        .drop(columns=['_sig']).reset_index(drop=True))
print("\nHOLDINGS (sorted by proximity to exit):")
print(h_df.to_string(index=False))

## Equity Curve

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')
cum_port = (1+dl['ret_port']).cumprod()-1
cum_spy  = (1+dl['ret_spy']).cumprod()-1
ax.plot(dl['date'], cum_port*100, color=C['gold'], linewidth=1.5,
        label='BT10', zorder=3)
ax.plot(dl['date'], cum_spy*100,  color=C['grey'], linewidth=1.0,
        label='SPY',  zorder=2)
ax.axhline(0, color='#30363d', linewidth=0.5, linestyle='--')
ax.fill_between(dl['date'], cum_port*100, 0,
                where=cum_port>=0, alpha=0.08, color=C['gold'])
ax.set_title(f"BT10 Long vs SPY  |  Since {port['inception_date']}  "
             f"|  Return on capital: {tot_ret:.1%}",
             color='#e6edf3', fontsize=12)
ax.set_ylabel("Cumulative Return (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha='right')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG['base_dir'],'equity_curve.png'),
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## Risk Metrics

In [ ]:
r_all = dl['ret_port'].values[1:]; s_all = dl['ret_spy'].values[1:]
n = len(r_all)
if n >= 5:
    # Total return always on capital deployed
    tot_r   = (port['total_value']-port['inception_value'])/port['inception_value']
    ann_r   = float((1+tot_r)**(252/n)-1)
    ann_v   = float(r_all.std()*np.sqrt(252))
    sharpe  = ann_r/ann_v if ann_v > 0 else None
    dn_r    = r_all[r_all < 0]
    sortino = ann_r/(dn_r.std()*np.sqrt(252)) if len(dn_r)>1 else None
    cum     = np.cumprod(1+r_all)
    mdd     = float(np.max(1-cum/np.maximum.accumulate(cum)))
    beta    = float(np.cov(r_all,s_all)[0,1]/np.var(s_all)) \
              if np.var(s_all) > 0 else None
    tr      = pd.read_csv(CFG['f_trades']) if os.path.exists(CFG['f_trades']) \
              else pd.DataFrame()
    n_trail = (tr['exit_type']=='TRAIL').sum() if not tr.empty else 0
    n_cusum = (tr['exit_type']=='CUSUM').sum() if not tr.empty else 0
    print("\nRISK METRICS:")
    print(f"  Total Return:      {tot_r:.1%}")
    print(f"  Ann. Return (est): {ann_r:.1%}")
    print(f"  Ann. Volatility:   {ann_v:.1%}")
    print(f"  Sharpe (est):      {sharpe:.3f}" if sharpe else "  Sharpe: —")
    print(f"  Sortino (est):     {sortino:.3f}" if sortino else "  Sortino: —")
    print(f"  Max Drawdown:      {-mdd:.1%}")
    print(f"  Beta vs SPY:       {beta:.3f}" if beta else "  Beta: —")
    print(f"  Days in market:    {n}")
    print(f"  Exits — Trail:     {n_trail}")
    print(f"  Exits — CUSUM:     {n_cusum}")

## RS Charts
*One chart per holding, sorted by proximity to exit signal.*

In [ ]:
def plot_rs_chart(sym, h):
    last_ts  = pd.Timestamp(LAST_DATE)
    lb_start = last_ts - pd.DateOffset(days=400)
    if sym not in px.columns: return
    sub = px.loc[lb_start:last_ts, [sym, CFG['benchmark']]].dropna()
    if len(sub) < 30: return
    rs    = (sub[sym]/sub[CFG['benchmark']]).values
    dates = sub.index
    n_use = min(CFG['lb_1y'], len(rs))
    rs1y  = rs[-n_use:]; d1y = dates[-n_use:]
    t     = np.arange(len(rs1y), dtype=float)
    b, a  = np.polyfit(t, rs1y, 1)
    trend = a + b*t
    res   = rs1y - trend
    se    = float(np.sqrt(np.sum(res**2)/max(len(rs1y)-2, 1)))
    ei    = np.searchsorted([d.date() for d in d1y],
                             date.fromisoformat(h['entry_date']))
    ei    = min(ei, len(rs1y)-1)
    ref   = rs1y[ei] if rs1y[ei] != 0 else 1.0
    ms    = max(h['cusum_pct'], h['trail_pct_to_fire'])
    col   = (C['red']  if ms >= CFG['cusum_alert'] else
             C['gold'] if ms >= CFG['cusum_warn']  else C['green'])
    fig   = plt.figure(figsize=(13, 4.5), facecolor='#0d1117')
    gs    = GridSpec(2, 1, height_ratios=[2,1], hspace=0.05)
    ax1   = fig.add_subplot(gs[0])
    ax2   = fig.add_subplot(gs[1], sharex=ax1)
    for ax in [ax1, ax2]:
        ax.set_facecolor('#161b22'); ax.tick_params(colors='#8b949e')
        ax.spines[:].set_edgecolor('#30363d'); ax.grid(True,color='#21262d',alpha=0.4)
    ax1.fill_between(d1y, (trend-CFG['sd_thresh']*se)/ref,
                          (trend+CFG['sd_thresh']*se)/ref,
                     color='#1f6feb', alpha=0.12)
    ax1.plot(d1y, (trend+CFG['sd_thresh']*se)/ref,
             color=C['gold'], lw=0.5, linestyle='dotted')
    ax1.plot(d1y, (trend-CFG['sd_thresh']*se)/ref,
             color=C['red'],  lw=0.5, linestyle='dotted')
    ax1.plot(d1y, trend/ref, color='#1f6feb', lw=1.0)
    ax1.plot(d1y, rs1y/ref,  color='#e6edf3', lw=1.0)
    ax1.axhline(1, color='#8b949e', lw=0.3, linestyle='--')
    if ei < len(d1y):
        ax1.axvline(d1y[ei], color=C['green'], lw=0.8, linestyle='--')
    ax1.set_title(
        f"{sym}  |  z={h['z_now']:.2f}  |  "
        f"CUSUM={h['cusum_pct']:.0f}%  "
        f"Trail={h['trail_pct_to_fire']:.0f}%  "
        f"PeakEx={h['peak_excess']:.1%}  "
        f"{h['days_held']}d",
        color=col, fontsize=10)
    ax1.set_ylabel("RS (1.0=entry)", fontsize=8)
    plt.setp(ax1.get_xticklabels(), visible=False)
    zh = h['cusum_z_hist']; n_zh = len(zh)
    if n_zh >= 2:
        S = np.zeros(n_zh)
        for k, z in enumerate(zh):
            pv = S[k-1] if k > 0 else 0.0
            S[k] = min(0.0, pv+z+CFG['cusum_k']) \
                   if z < -CFG['cusum_gate'] else 0.0
        all_td = [d.date() for d in
                  px.loc[pd.Timestamp(h['entry_date']):last_ts].index]
        cd = [pd.Timestamp(d) for d in
              (all_td[-n_zh:] if len(all_td) >= n_zh else all_td)[:len(S)]]
        S  = S[:len(cd)]
        pct_f = round(abs(S[-1])/CFG['cusum_h']*100, 1)
        ax2.fill_between(cd, S, 0, alpha=0.18, color=C['red'])
        ax2.plot(cd, S, color=col, lw=0.9)
        ax2.axhline(-CFG['cusum_h'], color=C['gold'], lw=0.8, linestyle='--')
        ax2.axhline(0, color='#8b949e', lw=0.3)
        ax2.text(0.01, 0.05, f"Fire: -{CFG['cusum_h']}",
                 transform=ax2.transAxes, color=C['gold'], fontsize=7)
        ax2.text(0.99, 0.85, f"{pct_f}% to CUSUM fire",
                 transform=ax2.transAxes, ha='right',
                 color=col, fontsize=8, fontweight='bold')
    else:
        ax2.text(0.5, 0.5, 'Accumulating...', ha='center',
                 transform=ax2.transAxes, color='#8b949e')
    ax2.set_ylabel("CUSUM", fontsize=8)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax2.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax2.get_xticklabels(), rotation=30, ha='right', fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(CFG['base_dir'], f"rs_{sym}.png"),
                dpi=120, bbox_inches='tight', facecolor='#0d1117')
    plt.show(); plt.close()
 
print("Plotting RS charts...")
for sym, h in sorted(port['holdings'].items(),
    key=lambda x: max(x[1]['cusum_pct'],
                      x[1]['trail_pct_to_fire']), reverse=True):
    plot_rs_chart(sym, h)

## Top 100 Rankings

In [ ]:
print("\nTOP 100 LONG CANDIDATES:")
print(f"{'Rk':>3} {'Ticker':<7} {'Sector':<25} {'Score':>6} "
      f"{'Z':>6} {'ExVol':>7} {'Thresh':>7} {'Price':>8}")
print("-" * 75)
for i, row in cand_sc.head(100).iterrows():
    held_m = '*' if row['ticker'] in port['holdings'] else ' '
    thresh = CFG['trail_n']*row['ann_ex_vol']
    print(f"{i+1:>3}{held_m} {row['ticker']:<7} {row['sector']:<25} "
          f"{row['composite']:>6.4f} {row['z_now']:>6.2f} "
          f"{row['ann_ex_vol']:>7.1%} {thresh:>7.1%} "
          f"{row['last_px']:>8.2f}")

## Trade Log

In [ ]:
if os.path.exists(CFG['f_trades']):
    tr = pd.read_csv(CFG['f_trades'])
    print(f"\nTRADE LOG (last 50 of {len(tr)} total):")
    print(tr.tail(50).to_string(index=False))
else:
    print("No trades yet.")

In [ ]:
# ── Performance Attribution ───────────────────────────────────────────────────
import pandas as pd
import numpy as np

trades = pd.read_csv(CFG['f_trades'])

buys  = trades[trades['action'] == 'BUY'].sort_values('date').reset_index(drop=True)
sells = trades[trades['action'] == 'SELL'].sort_values('date').reset_index(drop=True)

# ── Reconstruct round-trip P&L ────────────────────────────────────────────────
long_trades = []
for _, sell in sells.iterrows():
    sym   = sell['ticker']
    prior = buys[(buys['ticker'] == sym) & (buys['date'] <= sell['date'])]
    if prior.empty:
        continue
    buy       = prior.iloc[-1]
    shares    = sell['shares']
    buy_val   = buy['price']  * shares
    sell_val  = sell['price'] * shares
    pnl       = sell_val - buy_val
    pnl_pct   = pnl / buy_val
    hold_days = (pd.Timestamp(sell['date']) - pd.Timestamp(buy['date'])).days
    long_trades.append(dict(
        ticker=sym, sector=sell.get('sector', ''),
        entry_date=buy['date'], exit_date=sell['date'],
        hold_days=hold_days, shares=shares,
        entry_px=buy['price'], exit_px=sell['price'],
        pnl=pnl, pnl_pct=pnl_pct,
        exit_type=sell.get('exit_type', '')))

lt = pd.DataFrame(long_trades)

# ── Summary stats ─────────────────────────────────────────────────────────────
def trade_stats(df, label):
    if df.empty:
        print(f"\n{label}: no closed trades.")
        return
    wins = df[df['pnl'] > 0]
    loss = df[df['pnl'] <= 0]
    total_pnl = df['pnl'].sum()
    win_rate  = len(wins) / len(df)
    avg_hold  = df['hold_days'].mean()
    pf        = wins['pnl'].sum() / abs(loss['pnl'].sum()) if len(loss) > 0 else np.inf

    print(f"""
{'─'*55}
  {label}
{'─'*55}
  Closed trades     : {len(df)}
  Total P&L         : ${total_pnl:>12,.2f}
  Win rate          : {win_rate:.1%}  ({len(wins)}W / {len(loss)}L)
  Profit factor     : {pf:.2f}
  Avg hold (days)   : {avg_hold:.1f}

  Wins
    Avg             : {wins['pnl_pct'].mean():.2%}  (${wins['pnl'].mean():,.0f})
    Median          : {wins['pnl_pct'].median():.2%}  (${wins['pnl'].median():,.0f})
    Largest         : {wins['pnl_pct'].max():.2%}  ({wins.loc[wins['pnl_pct'].idxmax(), 'ticker']})

  Losses
    Avg             : {loss['pnl_pct'].mean():.2%}  (${loss['pnl'].mean():,.0f})
    Median          : {loss['pnl_pct'].median():.2%}  (${loss['pnl'].median():,.0f})
    Largest         : {loss['pnl_pct'].min():.2%}  ({loss.loc[loss['pnl_pct'].idxmin(), 'ticker']})""")

    # Exit type breakdown
    if 'exit_type' in df.columns and df['exit_type'].notna().any():
        et = (df.groupby('exit_type')
                .agg(n=('pnl', 'count'),
                     total_pnl=('pnl', 'sum'),
                     avg_pnl_pct=('pnl_pct', 'mean'),
                     win_rate=('pnl', lambda x: (x > 0).mean()))
                .sort_values('n', ascending=False))
        print(f"\n  By exit type:")
        print(f"  {'Type':<18} {'N':>4}  {'Total $':>10}  {'Avg %':>7}  {'Win%':>6}")
        print(f"  {'─'*18} {'─'*4}  {'─'*10}  {'─'*7}  {'─'*6}")
        for etype, row in et.iterrows():
            print(f"  {str(etype):<18} {int(row['n']):>4}  "
                  f"${row['total_pnl']:>9,.0f}  "
                  f"{row['avg_pnl_pct']:>6.1%}  "
                  f"{row['win_rate']:>5.1%}")

    # Top 5 winners / losers
    print(f"\n  Top 5 winners:")
    for _, r in wins.nlargest(5, 'pnl').iterrows():
        print(f"    {r['ticker']:<6}  {r['pnl_pct']:>7.2%}  ${r['pnl']:>9,.0f}  "
              f"{r['hold_days']}d  [{r.get('exit_type', '')}]")
    print(f"\n  Top 5 losers:")
    for _, r in loss.nsmallest(5, 'pnl').iterrows():
        print(f"    {r['ticker']:<6}  {r['pnl_pct']:>7.2%}  ${r['pnl']:>9,.0f}  "
              f"{r['hold_days']}d  [{r.get('exit_type', '')}]")

    # Sector breakdown
    if 'sector' in df.columns and df['sector'].notna().any():
        sec = (df.groupby('sector')
                 .agg(n=('pnl', 'count'),
                      total_pnl=('pnl', 'sum'),
                      avg_pnl_pct=('pnl_pct', 'mean'))
                 .sort_values('total_pnl', ascending=False))
        print(f"\n  By sector:")
        print(f"  {'Sector':<30} {'N':>4}  {'Total $':>10}  {'Avg %':>7}")
        print(f"  {'─'*30} {'─'*4}  {'─'*10}  {'─'*7}")
        for sec_name, row in sec.iterrows():
            print(f"  {str(sec_name):<30} {int(row['n']):>4}  "
                  f"${row['total_pnl']:>9,.0f}  "
                  f"{row['avg_pnl_pct']:>6.1%}")


# ── Overall summary ───────────────────────────────────────────────────────────
inc_val   = CFG['initial_capital']
cur_val   = port['daily_log'][-1]['tot_value'] if port.get('daily_log') else inc_val
total_ret = (cur_val - inc_val) / inc_val

print(f"""
{'='*55}
  PERFORMANCE ATTRIBUTION  |  {port.get('last_signal_date', '')}
{'='*55}
  Inception capital : ${inc_val:>12,.2f}
  Current value     : ${cur_val:>12,.2f}
  Total return      : {total_ret:>11.2%}
  Closed trade P&L  : ${lt['pnl'].sum() if not lt.empty else 0:>12,.2f}
{'='*55}""")

trade_stats(lt, "LONG ONLY  —  closed round-trips")

# ── Open positions ────────────────────────────────────────────────────────────
today_ts = pd.Timestamp(LAST_DATE)
print(f"\n{'─'*55}")
print(f"  OPEN POSITIONS")
print(f"{'─'*55}")
print(f"  {'Ticker':<7} {'Entry':>10}  {'Cur Px':>8}  {'Unreal%':>8}  Days")
print(f"  {'─'*7} {'─'*10}  {'─'*8}  {'─'*8}  {'─'*4}")

for sym, h in sorted(port.get('holdings', {}).items()):
    try:
        cur_px = float(px.loc[today_ts, sym]) if sym in px.columns else h['avg_cost']
    except:
        cur_px = h['avg_cost']
    unreal = (cur_px - h['avg_cost']) / h['avg_cost']
    days   = (LAST_DATE - date.fromisoformat(h['entry_date'])).days
    print(f"  {sym:<7} {h['avg_cost']:>10.2f}  {cur_px:>8.2f}  {unreal:>8.2%}  {days}")

## Email Alert

In [ ]:
def send_email_alert(port, cand_sc):
    if not CFG['send_email']:
        print("[EMAIL SUPPRESSED — set send_email=True in CFG to enable]")
        return
    tot_r      = (port['total_value']-port['inception_value'])/port['inception_value']
    near_exit  = sorted(port['holdings'].items(),
        key=lambda x: max(x[1]['cusum_pct'],x[1]['trail_pct_to_fire']),
        reverse=True)[:5]
    lines = [f"BT10 Daily — {LAST_DATE}",
             f"Portfolio: ${port['total_value']:,.0f}  Return: {tot_r:.1%}", ""]
    if port['pending_sells']:
        lines.append(f"SELL TODAY: {', '.join(port['pending_sells'])}")
    if port['pending_buys']:
        lines.append(f"BUY TODAY:  {', '.join(port['pending_buys'])}")
    lines += ["", "NEAREST TO EXIT:"]
    for sym, h in near_exit:
        lines.append(f"  {sym}: CUSUM={h['cusum_pct']:.0f}%  "
                     f"Trail={h['trail_pct_to_fire']:.0f}%")
    body    = "\n".join(lines)
    subject = (f"BT10 | {LAST_DATE} | {tot_r:.1%}" +
               (f" | SELL: {','.join(port['pending_sells'])}"
                if port['pending_sells'] else ""))
    try:
        msg = MIMEMultipart()
        msg['From']=CFG['email_from']; msg['To']=CFG['email_to']
        msg['Subject']=subject; msg.attach(MIMEText(body,'plain'))
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as s:
            s.login(CFG['email_from'], CFG['email_password'])
            s.send_message(msg)
        print(f"Email sent: {subject}")
    except Exception as e:
        print(f"Email failed: {e}")
 
send_email_alert(port, cand_sc)
print("\n" + "="*60)
print(f"  BT10 run complete — {LAST_DATE}")
print("="*60)